# Tutorial 00 (Lung, Demo): Predict Lung Motion From a Single Chest CT

## Purpose

A self-contained demo of the lung motion-prediction pipeline that fetches
every input it needs, so it runs with nothing more than a clone of the
repository:

1. Download the ``Chest-CT`` sample volume (an ungated 3-D chest CT) and the
   pretrained ``physicsnemo_mgn_lung_motion`` MeshGraphNet checkpoint with
   ``monai_physio.DownloadData``. The checkpoint directory also carries the
   lung PCA shape model (``pca_model.json``, ``pca_mean_surface.vtp``) that
   network was trained against, so no separate Tutorial 6 run is needed.
2. Segment the lungs with ``TCIA_LUNG_4D_CT.segmenter`` (``SegmentNVSegmentCTMRI``)
   and extract their surface.
3. Fit that PCA shape model to the surface with
   ``WorkflowFitStatisticalModelToPatient``, giving this patient's PCA
   coefficients and fitted SSM surface.
4. Predict lung motion across a demo respiratory-stage grid with the
   pretrained MeshGraphNet, carry the CT through each stage's deformation,
   and write the series as VTP surfaces and one animated USD --
   ``WorkflowInferMovement.process_time_series``.

``Chest-CT`` is a single, ungated volume -- unlike ``TCIA-4DLung``'s gated
sequences, it has no ``g{PPP}`` respiratory phases to parse a stage grid
from. This demo instead predicts motion across an illustrative grid of
normalized stages, so the same pretrained network that predicts a real
respiratory cycle in ``tutorial_12_infer_respiratory_motion_tcia_lung_4d_ct.py`` can be
shown working on any chest CT.

## Data Required

Nothing -- the ``Chest-CT`` volume, the pretrained MeshGraphNet weights and
the PCA shape model they were trained against are all downloaded by this
notebook.

## Outputs (under ``output/tutorial_00_infer_respiratory_motion_tcia_lung_3d_ct/``)

  * ``Chest-CT_lung_labelmap.nii.gz`` -- segmentation
  * ``Chest-CT_surface.json`` -- this patient's shape parameters
  * ``Chest-CT_ssm_surface.vtp`` -- the model fitted to the patient
  * ``Chest-CT_surface_s{TTT}_pred.vtp`` -- predicted surface per stage
  * ``Chest-CT_s{TTT}_pred.mha`` -- CT carried to that stage
  * ``Chest-CT_pred.usd`` -- animated predicted motion

In [ ]:
# Imports
import json
import logging
from typing import cast

import itk
import numpy as np
import pyvista as pv
from parameters_tcia_lung_4d_ct import TCIA_LUNG_4D_CT

from monai_physio import (
    DownloadData,
    MONAIPhysioBase,
    ProcessSurfaceMesh,
    WorkflowConvertImageToVTK,
    WorkflowFitStatisticalModelToPatient,
    WorkflowInferMovement,
    WorkflowInferPhysicsNeMo,
)

In [ ]:
# Data directory specification
test_mode = False

case_id = "Chest-CT"

data_dir = TCIA_LUNG_4D_CT.hold_out_directory(test_mode)
weights_dir = TCIA_LUNG_4D_CT.weights_directory(test_mode)
output_dir = (
    TCIA_LUNG_4D_CT.output_directory(test_mode)
    / "tutorial_00_infer_respiratory_motion_tcia_lung_3d_ct"
)

# Weights Tutorial 9 trains; this demo downloads a pretrained copy instead.
# The PCA model and mean surface it was trained against ship in the same
# checkpoint directory.
model_dir = TCIA_LUNG_4D_CT.mgn_weights_directory(test_mode)
epoch = None  # None uses the final weights
pca_model_file = model_dir / "pca_model.json"
pca_mean_file = model_dir / "pca_mean_surface.vtp"

number_of_pca_components = TCIA_LUNG_4D_CT.pca_components(test_mode)

# time points to reconstruct
stages = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# Gaussian sigma, in mm, that spreads the predicted surface displacements
# into the continuous field the CT is resampled through.
smoothing_sigma_mm = 10.0

log_level = logging.INFO
logging.basicConfig(level=log_level)
logger = logging.getLogger("tutorial_00_infer_respiratory_motion_tcia_lung_3d_ct")

output_dir.mkdir(parents=True, exist_ok=True)

## Step 1: Download the Chest-CT volume and the pretrained MeshGraphNet weights

In [ ]:
patient_image_file = DownloadData.DownloadChestCTData(data_dir)
DownloadData.DownloadPhysicsNeMoMGNLungMotionData(weights_dir)

if not (model_dir / "mgn_stage_model.pt").exists():
    raise FileNotFoundError(
        f"Pretrained MeshGraphNet checkpoint not found: {model_dir}"
    )
if not (pca_model_file.exists() and pca_mean_file.exists()):
    raise FileNotFoundError(
        f"PCA shape model not found alongside checkpoint: {model_dir}"
    )

pca_mean_surface = cast(pv.DataSet, pv.read(str(pca_mean_file)))
with pca_model_file.open(encoding="utf-8") as f:
    pca_model = json.load(f)

patient_image = itk.imread(str(patient_image_file))
logger.info("Loaded %s", patient_image_file)

## Step 2: Segment the lungs

In [ ]:
lung_labelmap_file = output_dir / f"{case_id}_lung_labelmap.nii.gz"

segmentation_method = TCIA_LUNG_4D_CT.segmenter(test_mode, log_level=log_level)
segmentation_result = WorkflowConvertImageToVTK(
    segmentation_method=segmentation_method,
    log_level=log_level,
).process(
    input_image=patient_image,
    anatomy_groups=[TCIA_LUNG_4D_CT.anatomy_group],
    surface_reduction_rate=TCIA_LUNG_4D_CT.surface_reduction_rate,
    extract_label_surfaces=True,
    extract_group_surfaces=False,
)
lung_surface = ProcessSurfaceMesh.combine_surfaces(
    segmentation_result["label_surfaces"]
)
# anatomy_groups only selects which structures are contoured into
# label_surfaces above; the labelmap itself is the segmenter's full,
# unfiltered whole-body output (ribs, spine, heart, etc. included).
lung_labelmap = segmentation_result["labelmap"]
itk.imwrite(lung_labelmap, str(lung_labelmap_file), compression=True)
logger.info("Segmented lungs in %s", patient_image_file.name)

# Isolate just the lung labels into a binary mask, the same way
# tutorial_13_combine_heart_and_lung_motion_chest_3d_ct.py's binary_mask_on_grid does: this (not
# the whole-body lung_labelmap above) is what the fit's distance map and
# exterior_mask must be built from. Built from the whole body, the fit would
# score every other structure as on-target, and nearly every voxel in the
# torso would read as "interior" and get the full, unrestricted deformation
# instead of only the lung's push and pull.
lung_label_ids = list(segmentation_method.taxonomy.labels_in_group("lung"))
lung_only_mask_arr = np.isin(
    itk.array_from_image(lung_labelmap), lung_label_ids
).astype(np.uint8)
lung_only_mask = itk.image_from_array(lung_only_mask_arr)
lung_only_mask.CopyInformation(lung_labelmap)

# Release the model before fitting; segmentation is finished in this notebook.
segmentation_method.release_resources(empty_cuda_cache=True)

## Step 3: Fit the lung shape model to the patient

The PCA coefficients are what the network is conditioned on; the fitted
surface is what its displacements are added to.

In [ ]:
fit_workflow = WorkflowFitStatisticalModelToPatient(
    template_model=pca_mean_surface,
    patient_models=[lung_surface],
    patient_image=patient_image,
    patient_labelmap=lung_only_mask,
    log_level=log_level,
)
fit_workflow.set_use_pca_registration(
    use_pca_registration=True,
    pca_model=pca_model,
    number_of_pca_components=number_of_pca_components,
    use_surface=False,
)
fit_workflow.set_icp_transform_type(TCIA_LUNG_4D_CT.icp_transform_type)
fit_workflow.set_mask_dilation_mm(TCIA_LUNG_4D_CT.mask_dilation_mm)
fit_workflow.set_distancemap_squared_max(TCIA_LUNG_4D_CT.distancemap_squared_max)
# A progress callback receives the fraction of the fit completed and a
# message naming the step; this one prints every tenth of the way.
printed_tenths: set[int] = set()


def show_progress(fraction: float, message: str) -> None:
    tenth = int(fraction * 10)
    if tenth not in printed_tenths:
        printed_tenths.add(tenth)
        print(f"{fraction:6.1%} {message}")


# The callback is shared: every MONAI Physio class reports to it from now on.
MONAIPhysioBase.set_progress_callback(show_progress)
fit_result = fit_workflow.process()

pca_coefficients = fit_workflow.pca_coefficients
assert pca_coefficients is not None
# WorkflowInferMovement.process_time_series names every predicted surface
# after this file's stem, so naming it "{case_id}_surface" is what gives the
# predicted surfaces their "{case_id}_surface_s{TTT}_pred.vtp" names below.
pca_coefficients_file = output_dir / f"{case_id}_surface.json"
with pca_coefficients_file.open(mode="w", encoding="utf-8") as f:
    json.dump(pca_coefficients.tolist(), f)

# The lung PCA model is built from surfaces only, so the model *is* a
# surface here: only the .vtp is written.
fitted_reference_mesh_file = output_dir / f"{case_id}_ssm_surface.vtp"
fit_result["fitted_reference_mesh"].save(str(fitted_reference_mesh_file))
logger.info("Fitted the lung model to %s", patient_image_file.name)

## Step 4: Predict lung motion with the pretrained MeshGraphNet

Predict every demo stage, warp the reference CT through each stage's
deformation, and write the animated USD. -1000 HU is air, the value a CT
grid samples outside itself. `exterior_mask=lung_labelmap` restricts the warp
outside the lungs to the surface-normal component, fading to zero by
`exterior_falloff_distance_mm`, so the chest wall and mediastinum are pushed
and pulled by the breathing lung rather than dragged along with it.

In [ ]:
infer_workflow = WorkflowInferPhysicsNeMo(
    model_directory=model_dir, epoch=epoch, log_level=log_level
)
infer_result = WorkflowInferMovement(
    infer_workflow, log_level=log_level
).process_time_series(
    shape_parameters=pca_coefficients_file,
    stages=stages,
    output_directory=output_dir,
    fitted_reference_mesh=fitted_reference_mesh_file,
    reference_image=patient_image,
    warp_interpolation="linear",
    warp_background_value=-1000.0,
    smoothing_sigma_mm=smoothing_sigma_mm,
    usd_project_name=f"{case_id}_pred",
    anatomy_type="lung",
    separate_by_connectivity=True,
    exterior_mask=lung_only_mask,
    exterior_falloff_distance_mm=10.0,
    return_transforms=False,  # Saved images and meshes are sufficient here.
)

# process_time_series names the warped CT "{stem}_s{TTT}_warped.mha"; rename
# each to this notebook's "{case_id}_s{TTT}_pred.mha" convention.
warped_images = []
for stage, warped_file in zip(stages, infer_result["warped_images"]):
    tag = f"s{int(stage * 100):03d}"
    renamed_file = output_dir / f"{case_id}_{tag}_pred.mha"
    warped_file.replace(renamed_file)
    warped_images.append(renamed_file)
infer_result["warped_images"] = warped_images

logger.info("USD: %s", infer_result["usd_file"])

## Step 5: Look at the fitted and predicted surfaces

In [ ]:
pv.set_jupyter_backend("static")

plotter = pv.Plotter(shape=(1, 2), window_size=[1000, 500])
plotter.subplot(0, 0)
plotter.add_mesh(
    cast(pv.DataSet, pv.read(str(fitted_reference_mesh_file))),
    color="steelblue",
)
plotter.add_text("Fitted reference surface", font_size=10)
plotter.camera_position = "iso"

plotter.subplot(0, 1)
plotter.add_mesh(
    cast(pv.DataSet, pv.read(str(infer_result["predicted_surfaces"][0]))),
    color="limegreen",
)
plotter.add_text(f"Predicted surface, stage {stages[0]}", font_size=10)
plotter.camera_position = "iso"

plotter.show()